# Анализ продаж на маркетплейсе Amazon (США)

**Цель исследования:** исследовать продажи, ассортимент и поведение клиентов на маркетплейсе Amazon в США, выявить ключевые товары, категории, регионы и закономерности, важные для продуктовой и коммерческой команды.

**Роль:** кейс для портфолио продуктового/бизнес‑аналитика в e‑commerce и маркетплейсах.

**Данные:** обезличенные данные по заказам в США за 2020 год: дата заказа, способ доставки, сегмент клиента, регион, штат, город, категория и подкатегория товара, название товара, продажи (Sales), количество (Quantity), скидка (Discount) и прибыль (Profit).

## Цель исследования и описание данных

В этом кейсе выполняется подробный EDA по заказам, выручке, географии, ассортименту, скидкам и сегментам клиентов. Результат оформлен как слайд‑дека с выводами и рекомендациями для e‑commerce.

## Импорт библиотек и загрузка данных

Подключаем библиотеки, задаём единую сине‑фиолетовую палитру для маркетплейс‑кейса, загружаем CSV.

In [ ]:
# Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Единая палитра для Amazon/маркетплейс‑кейса (синие/фиолетовые оттенки)
sns.set_theme(style="whitegrid", palette="Blues")
pd.set_option("display.max_columns", 50)

# Загрузка данных (в Colab — загрузите файл и укажите путь)
FILE_PATH = "amazon_market_data.csv"
df = pd.read_csv(FILE_PATH)

df.head()
df.info()

## Первичный обзор, очистка и подготовка

Приведение имён колонок к snake_case, преобразование дат и числовых полей, проверка дубликатов и пропусков.

In [ ]:
# Имена колонок: нижний регистр, пробелы и дефисы — в подчёркивания
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)
df.columns.tolist()

In [ ]:
# Даты: в CSV формат dd-mm-yy
df["order_date"] = pd.to_datetime(df["order_date"], format="%d-%m-%y", errors="coerce")

# Числовые поля
for col in ["sales", "quantity", "discount", "profit"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df.dtypes

In [ ]:
# Дубликаты и пропуски
dup = df.duplicated().sum()
print(f"Полных дубликатов строк: {dup}")
if dup > 0:
    df = df.drop_duplicates()

missing = df.isna().mean().sort_values(ascending=False)
print("\nДоля пропусков:")
print(missing[missing > 0].round(4).to_string() if (missing > 0).any() else "Пропусков нет")

# Удаление строк с критическими пропусками (дата, выручка, категория)
critical = [c for c in ["order_date", "sales", "category"] if c in df.columns]
df = df.dropna(subset=critical)
print(f"\nСтрок после очистки: {len(df)}")

**Вывод:** Имена колонок приведены к snake_case. Даты и числовые поля преобразованы к нужным типам. Дубликаты удалены; пропуски оценены, строки с пропусками в ключевых полях отфильтрованы. Данные готовы к расчёту метрик и визуализациям.

## Базовые метрики: заказы, клиенты, период, выручка

Оцениваем объём выборки: уникальные заказы и клиенты, период, суммарные продажи и прибыль, средний чек.

In [ ]:
n_orders = df["order_id"].nunique()
n_customers = df["customer_id"].nunique()
date_min, date_max = df["order_date"].min(), df["order_date"].max()
total_sales = df["sales"].sum()
total_profit = df["profit"].sum()
avg_order_sales = df.groupby("order_id")["sales"].sum().mean()

print(f"Уникальных заказов:     {n_orders:,}")
print(f"Уникальных клиентов:   {n_customers:,}")
print(f"Период заказов:        {date_min.date()} — {date_max.date()}")
print(f"Суммарные продажи:     {total_sales:,.2f}")
print(f"Суммарная прибыль:     {total_profit:,.2f}")
print(f"Средний чек (на заказ): {avg_order_sales:.2f}")
neg_profit = (df["profit"] < 0).sum()
print(f"Строк с отрицательной прибылью: {neg_profit}")

**Вывод:** По метрикам виден масштаб выборки (заказы, клиенты) и период. Суммарные продажи и прибыль задают объём бизнеса; средний чек характеризует средний размер заказа. Наличие строк с отрицательной прибылью указывает на убыточные позиции или заказы (скидки, себестоимость) — их стоит учитывать в анализе скидок и маржинальности.

## География: регионы, штаты, города

Распределение заказов и выручки по регионам, топ штатов и городов по продажам.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

by_region_orders = df.groupby("region")["order_id"].nunique().sort_values(ascending=False)
by_region_orders.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Количество заказов по регионам")
axes[0].set_xlabel("Регион")
axes[0].set_ylabel("Количество заказов")
axes[0].tick_params(axis="x", rotation=45)

by_region_sales = df.groupby("region")["sales"].sum().sort_values(ascending=False)
by_region_sales.plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Суммарные продажи по регионам")
axes[1].set_xlabel("Регион")
axes[1].set_ylabel("Продажи")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
by_state = df.groupby("state")["sales"].sum().nlargest(10)
by_city = df.groupby("city")["sales"].sum().nlargest(10)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
by_state.sort_values().plot(kind="barh", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Топ-10 штатов по сумме продаж")
axes[0].set_xlabel("Продажи")
axes[0].set_ylabel("Штат")
by_city.sort_values().plot(kind="barh", ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Топ-10 городов по сумме продаж")
axes[1].set_xlabel("Продажи")
axes[1].set_ylabel("Город")
plt.tight_layout()
plt.show()

**Вывод:** Ключевые регионы и топ штатов/городов по выручке задают географию сбыта. Концентрация продаж в нескольких регионах или штатах показывает, где сосредоточен основной объём и куда направлять логистику и маркетинг.

## Динамика количества заказов

Число уникальных заказов по месяцам.

In [ ]:
orders_daily = df.groupby("order_date")["order_id"].nunique()
orders_monthly = orders_daily.resample("M").sum()

fig, ax = plt.subplots(figsize=(10, 4))
orders_monthly.plot(ax=ax, color="steelblue", linewidth=2)
ax.set_title("Динамика количества заказов по месяцам")
ax.set_xlabel("Месяц")
ax.set_ylabel("Количество заказов")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Вывод:** Тренд по месяцам (рост, спад или плато) и возможная сезонность помогают оценивать активность спроса и планировать запасы и кампании.

## Динамика выручки (Sales)

Сумма продаж по месяцам и сравнение с динамикой числа заказов.

In [ ]:
sales_daily = df.groupby("order_date")["sales"].sum()
sales_monthly = sales_daily.resample("M").sum()

fig, axes = plt.subplots(2, 1, figsize=(10, 7))
sales_monthly.plot(ax=axes[0], color="steelblue", linewidth=2)
axes[0].set_title("Динамика выручки по месяцам")
axes[0].set_xlabel("Месяц")
axes[0].set_ylabel("Продажи")
axes[0].grid(True, alpha=0.3)

axes[1].plot(orders_monthly.index, orders_monthly.values, color="steelblue", label="Заказы")
ax2 = axes[1].twinx()
ax2.plot(sales_monthly.index, sales_monthly.values, color="darkorange", alpha=0.8, label="Выручка")
axes[1].set_title("Сравнение: количество заказов и выручка по месяцам")
axes[1].set_ylabel("Количество заказов")
ax2.set_ylabel("Выручка")
axes[1].legend(loc="upper left")
ax2.legend(loc="upper right")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Вывод:** Пики выручки и заказов по месяцам показывают, совпадают ли периоды высокой активности. Расхождение (много заказов при умеренной выручке или наоборот) указывает на периоды с более высоким или низким средним чеком.

## Категории и подкатегории товаров

Продажи по категориям и топ подкатегорий по выручке.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

by_cat = df.groupby("category")["sales"].sum().sort_values(ascending=False)
by_cat.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Продажи по категориям товаров")
axes[0].set_xlabel("Категория")
axes[0].set_ylabel("Продажи")
axes[0].tick_params(axis="x", rotation=0)

by_sub = df.groupby("sub_category")["sales"].sum().nlargest(20).sort_values()
by_sub.plot(kind="barh", ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Топ-20 подкатегорий по продажам")
axes[1].set_xlabel("Продажи")
axes[1].set_ylabel("Подкатегория")
plt.tight_layout()
plt.show()

**Вывод:** Лидирующие категории задают основу ассортимента; топ подкатегорий показывает, какие направления (например, Office Supplies, Furniture, Technology) дают основной вклад в выручку и на что опираться при закупках и продвижении.

## Популярные товары и ассортимент

Топ товаров по количеству заказов (строк) и по сумме продаж.

In [ ]:
by_product_count = df.groupby("product_name").size().nlargest(20)
by_product_sales = df.groupby("product_name")["sales"].sum().nlargest(20)

fig, axes = plt.subplots(2, 1, figsize=(10, 10))
by_product_count.sort_values().plot(kind="barh", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Топ-20 товаров по количеству заказов (позиций)")
axes[0].set_xlabel("Количество")
axes[0].set_ylabel("Товар")
by_product_sales.sort_values().plot(kind="barh", ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Топ-20 товаров по сумме продаж")
axes[1].set_xlabel("Продажи")
axes[1].set_ylabel("Товар")
plt.tight_layout()
plt.show()

**Вывод:** Товары с максимальным числом заказов часто отличаются от товаров с максимальной выручкой (дешёвые частые покупки vs дорогие единичные). Оба среза важны для управления ассортиментом и складом.

## Анализ скидок и их влияние на прибыль

Распределение скидок, связь уровня скидки с прибылью и маржинальностью.

In [ ]:
df["profit_margin"] = np.where(df["sales"] > 0, df["profit"] / df["sales"], np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["discount"], bins=20, ax=axes[0], color="steelblue")
axes[0].set_title("Распределение скидок")
axes[0].set_xlabel("Скидка")
axes[0].set_ylabel("Количество")
sns.scatterplot(data=df.sample(min(2000, len(df))), x="discount", y="profit", hue="discount", palette="Blues", ax=axes[1], legend=False)
axes[1].set_title("Прибыль vs скидка (выборка)")
axes[1].set_xlabel("Скидка")
axes[1].set_ylabel("Прибыль")
plt.tight_layout()
plt.show()

In [ ]:
# Бинны скидок и средняя маржа по ним
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.01]
labels = ["0–20%", "20–40%", "40–60%", "60–80%", "80%+"]
df["discount_bin"] = pd.cut(df["discount"], bins=bins, labels=labels, include_lowest=True)
margin_by_discount = df.groupby("discount_bin", observed=True)["profit_margin"].mean()

fig, ax = plt.subplots(figsize=(7, 4))
margin_by_discount.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Средняя прибыльность (profit/sales) по уровню скидки")
ax.set_xlabel("Уровень скидки")
ax.set_ylabel("Средняя маржа")
ax.axhline(0, color="gray", linestyle="--")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()
print(margin_by_discount.round(4).to_string())

**Вывод:** Высокие скидки в среднем снижают маржинальность; уровни скидок, при которых маржа уходит в минус или близка к нулю, — кандидаты на пересмотр ценовой и скидочной политики.

## Клиентские сегменты (Segment) и поведение

Распределение по сегментам (Consumer, Corporate, Home Office): заказы, выручка, средний чек, прибыль.

In [ ]:
seg_orders = df.groupby("segment")["order_id"].nunique()
seg_sales = df.groupby("segment")["sales"].sum()
seg_avg = df.groupby("segment")["sales"].mean()
seg_profit = df.groupby("segment")["profit"].sum()
seg_profit_avg = df.groupby("segment")["profit"].mean()

print("По сегментам:")
print(pd.DataFrame({
    "Заказов": seg_orders,
    "Сумма продаж": seg_sales.round(2),
    "Средний чек (позиции)": seg_avg.round(2),
    "Сумма прибыли": seg_profit.round(2),
    "Средняя прибыль на позицию": seg_profit_avg.round(2)
}).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
seg_sales.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Суммарные продажи по сегменту")
axes[0].set_xlabel("Сегмент")
axes[0].set_ylabel("Продажи")
axes[0].tick_params(axis="x", rotation=45)
seg_profit.plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Суммарная прибыль по сегменту")
axes[1].set_xlabel("Сегмент")
axes[1].set_ylabel("Прибыль")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

**Вывод:** Сегмент с наибольшей выручкой и прибылью задаёт приоритет для B2B/B2C‑стратегии. Более маржинальные сегменты (например, Corporate или Home Office) — кандидаты на развитие и таргетированные предложения.

## Глубже по географии и категориям

Связь региона и категории (heatmap продаж); регион × сегмент.

In [ ]:
cross_region_cat = pd.crosstab(df["region"], df["category"], values=df["sales"], aggfunc="sum")
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(cross_region_cat, annot=True, fmt=".0f", cmap="Blues", ax=ax)
ax.set_title("Продажи по регионам и категориям")
ax.set_xlabel("Категория")
ax.set_ylabel("Регион")
plt.tight_layout()
plt.show()

In [ ]:
cross_region_seg = pd.crosstab(df["region"], df["segment"], values=df["sales"], aggfunc="sum")
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(cross_region_seg, annot=True, fmt=".0f", cmap="Blues", ax=ax)
ax.set_title("Продажи: регион × сегмент")
ax.set_xlabel("Сегмент")
ax.set_ylabel("Регион")
plt.tight_layout()
plt.show()

**Вывод:** В части регионов доминируют определённые категории; сочетание регион × сегмент показывает, где сильнее B2B vs B2C. Это задаёт приоритеты для ассортимента и маркетинга по регионам.

## Итоговые бизнес‑выводы и рекомендации

**1. Общие результаты**
- Объём выборки: число уникальных заказов и клиентов, период данных. Суммарная выручка и прибыль задают масштаб. Часть заказов/позиций с отрицательной прибылью — зона внимания для цен и скидок.

**2. География**
- Ключевые регионы и топ штатов/городов по продажам показывают концентрацию выручки. Приоритеты: логистика, региональный маркетинг и расширение в сильных регионах.

**3. Ассортимент**
- Лидирующие категории и подкатегории по выручке — основа ассортиментной политики. Топ товары по количеству заказов и по выручке (часто разные списки) задают фокус по складским запасам и продвижению.

**4. Скидки и прибыль**
- Высокие скидки в среднем снижают маржинальность. Уровни скидок с отрицательной или нулевой маржой стоит пересмотреть: ограничить глубину скидок или повысить базовые цены на чувствительные к скидкам позиции.

**5. Сегменты клиентов**
- Сегмент с максимальной выручкой и прибылью (Consumer/Corporate/Home Office) задаёт приоритет монетизации. Более маржинальные сегменты — кандидаты на таргетированные кампании и спецпредложения.

**6. Рекомендации**
- Приоритизировать категории и регионы с наибольшей выручкой и маржой; усилить ассортимент и наличие в топ подкатегориях и топ товарах. Пересмотреть скидочную политику на уровнях скидок, где прибыль падает или уходит в минус. Развивать наиболее прибыльные клиентские сегменты и комбинации регион × сегмент. Дополнительно: ABC‑анализ товаров, анализ корзины (сопутствующие покупки), A/B‑тесты по скидкам и ценам.